# Fixed-Size Chunking

In [18]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
import uuid

In [30]:
def perform_fixed_size_chunking(document, chunk_size=1000, overlap=200):

    text_splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size = chunk_size,
        chunk_overlap = overlap,
    )

    documents = []

    # Split the text into chunks
    chunk = text_splitter.split_text(document)
    print(f"Document split into {len(chunk)} chunks")

    # Convert to Document objects with metadata
    
    for i, chunk in enumerate(chunk):
        doc = Document(
            page_content=chunk,
            metadata = {
                "chunk_id": uuid.uuid4(),
                "total_chunks": len(chunk),
                "chunk_type": "fixed-size"
                
            }
        )
        documents.append(doc)

    return documents

In [25]:
text = """
# Python for Embeddings in AI Applications

## Introduction

Embeddings are numerical vector representations of data such as text, images, audio, or documents. They enable machine learning models to understand semantic meaning and relationships between different pieces of information. Embeddings are a fundamental component of modern AI systems, including semantic search, Retrieval-Augmented Generation (RAG), recommendation engines, clustering, and AI agents.

Python has become the preferred language for working with embeddings due to its rich ecosystem of AI and machine learning libraries.

## What Are Embeddings?

An embedding converts unstructured data into a dense vector of floating-point numbers. Similar data points produce vectors that are close together in vector space.

For example:

* "Python is a programming language"
* "Python is used for software development"

These sentences will generate embedding vectors that are semantically similar.

Example embedding vector:

```python
[0.123, -0.456, 0.789, ..., 0.234]
```

Modern embedding models typically generate vectors with dimensions ranging from 384 to 3072.

## Popular Python Libraries for Embeddings

### 1. Sentence Transformers

One of the most widely used libraries for generating text embeddings.

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

text = "Python is great for AI development"
embedding = model.encode(text)

print(len(embedding))
```

### 2. OpenAI Embeddings

Generate high-quality embeddings using OpenAI models.

```python
from openai import OpenAI

client = OpenAI()

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Python for AI applications"
)

embedding = response.data[0].embedding
print(len(embedding))
```

### 3. Ollama Local Embeddings

Generate embeddings locally without sending data to external APIs.

```python
import ollama

response = ollama.embeddings(
    model="nomic-embed-text",
    prompt="Python is used in machine learning"
)

embedding = response["embedding"]
```

## Choosing an Embedding Model

When selecting an embedding model, consider:

| Factor           | Description                          |
| ---------------- | ------------------------------------ |
| Accuracy         | Semantic understanding quality       |
| Dimension Size   | Storage and computation requirements |
| Latency          | Embedding generation speed           |
| Cost             | API or infrastructure expenses       |
| Language Support | Multilingual capabilities            |
| Context Length   | Maximum input size                   |

Common choices:

* **all-MiniLM-L6-v2**: Fast and lightweight
* **BAAI/bge-large-en-v1.5**: High accuracy
* **text-embedding-3-small**: Cost-effective API option
* **text-embedding-3-large**: Best OpenAI embedding quality
* **nomic-embed-text**: Popular local embedding model

## Measuring Similarity

Embeddings are commonly compared using cosine similarity.

```python
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    [embedding1],
    [embedding2]
)

print(similarity[0][0])
```

A score closer to 1 indicates greater semantic similarity.

## Embeddings in RAG Systems

A typical Retrieval-Augmented Generation (RAG) workflow:

1. Split documents into chunks.
2. Generate embeddings for each chunk.
3. Store embeddings in a vector database.
4. Convert user queries into embeddings.
5. Perform similarity search.
6. Send relevant chunks to an LLM for response generation.

Popular vector databases:

* Chroma
* FAISS
* Qdrant
* Weaviate
* Milvus

## Best Practices

* Normalize text before embedding.
* Use chunking for large documents.
* Select embedding models based on your use case.
* Store metadata alongside embeddings.
* Benchmark retrieval performance regularly.
* Monitor embedding costs and latency.

## Conclusion

Embeddings are the foundation of modern AI search and retrieval systems. Python provides powerful libraries and frameworks that make it easy to generate, store, and utilize embeddings for applications such as semantic search, recommendation systems, AI agents, and RAG pipelines. Understanding how embeddings work and how to choose the right embedding model is a critical skill for AI and machine learning engineers.

"""

In [31]:
# from langchain_community.document_loaders import PyMuPDFLoader, PyPDFDirectoryLoader

# pdf_dir = PyPDFDirectoryLoader(
#     "../data/pdf/",
#     glob="**/*.pdf",
# )

# documents = pdf_dir.load()

print(type(text))

chunked_docs = perform_fixed_size_chunking(text)

print("\n---- Chunking Result-----")
print(f"Total Document: {len(text)}")
print(f"Total Chunks: {len(chunked_docs)}")


<class 'str'>
Document split into 6 chunks

---- Chunking Result-----
Total Document: 4339
Total Chunks: 6


In [32]:
chunked_docs[0].page_content

'# Python for Embeddings in AI Applications\n\n## Introduction\n\nEmbeddings are numerical vector representations of data such as text, images, audio, or documents. They enable machine learning models to understand semantic meaning and relationships between different pieces of information. Embeddings are a fundamental component of modern AI systems, including semantic search, Retrieval-Augmented Generation (RAG), recommendation engines, clustering, and AI agents.\n\nPython has become the preferred language for working with embeddings due to its rich ecosystem of AI and machine learning libraries.\n\n## What Are Embeddings?\n\nAn embedding converts unstructured data into a dense vector of floating-point numbers. Similar data points produce vectors that are close together in vector space.\n\nFor example:\n\n* "Python is a programming language"\n* "Python is used for software development"\n\nThese sentences will generate embedding vectors that are semantically similar.\n\nExample embeddin

# Semantic Chunking

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

In [38]:



def perform_semantic_chunking(documents, chunk_size=100, chunk_overlap=100):
    

    text_splitter = RecursiveCharacterTextSplitter(
        separators=['\n\n', '\n', ".", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function = len
    )

    semantic_chunks=text_splitter.split_text(documents)
    print(f"Document split into {len(semantic_chunks)} semantic chunks")

    print(semantic_chunks[0])

    # Determine section titles for enhanced metadata
    section_patterns = [
        r'^#+\s+(.+)$',      # Markdown headers
        r'^.+\n[=\-]{2,}$',  # Underlined headers
        r'^[A-Z\s]+:$'       # ALL CAPS section titles
    ]

    # convert to Document objects with enchanced metadata
    documents = []
    current_section = "Introduction"

    for chunk in semantic_chunks:

        chunk_lines = chunk.split('\n')

        for line in chunk_lines:
            for pattern in section_patterns:
                match = re.match(pattern, line, re.MULTILINE)
                if match:
                    current_section = match.group(0)
                    break
        
        # calculate sematic density (ratio of non-stopwords to total words)
        words = re.findall(r'\b\w+\b', chunk.lower())
        stopwords = ['the', 'and', 'is', 'of', 'to', 'a', 'in', 'that', 'it', 'with', 'as', 'for']
        content_words = [w for w in words if w not in stopwords]
        semantic_density = len(content_words) / max(1, len(words))

        doc = Document(
            page_content=chunk,
            metadata={
                "chunk_id": 1,
                "total_chunks": len(semantic_chunks),
                "chunk_size": len(chunk),
                "chunk_type": "semantic",
                "section": current_section,
                "semantic_density": round(semantic_density, 2)
            }
        )
        documents.append(doc)
    return documents

In [39]:

chunked_docs = perform_semantic_chunking(
    text,
    chunk_size=500,
    chunk_overlap=100
)

# Display results
print("\n----- CHUNKING RESULTS -----")
print(f"Total semantic chunks: {len(chunked_docs)}")

# Print an example chunk
print("\n----- EXAMPLE SEMANTIC CHUNK -----")
middle_chunk_idx = len(chunked_docs) // 2
example_chunk = chunked_docs[middle_chunk_idx]
print(f"Chunk {middle_chunk_idx} content ({len(example_chunk.page_content)} characters):")
print("-" * 40)
print(example_chunk.page_content)
print("-" * 40)
print(f"Metadata: {example_chunk.metadata}")

# Optional: Calculate section distribution for analysis
section_counts = {}
for doc in chunked_docs:
    section = doc.metadata["section"]
    section_counts[section] = section_counts.get(section, 0) + 1

print("\n----- SECTION DISTRIBUTION -----")
for section, count in section_counts.items():
    print(f"{section}: {count} chunks")

# For integration with Databricks embeddings
print("\nTo integrate with Databricks:")
print("1. Create embeddings using the Databricks embedding API:")
print("   from langchain_community.embeddings import DatabricksEmbeddings")
print("   embeddings = DatabricksEmbeddings(endpoint='databricks-bge-large-en')")
print("2. Store documents and embeddings in Delta table")
print("3. Create Vector Search index using the semantic metadata for filtering")

Document split into 10 semantic chunks
# Python for Embeddings in AI Applications

## Introduction

Embeddings are numerical vector representations of data such as text, images, audio, or documents. They enable machine learning models to understand semantic meaning and relationships between different pieces of information. Embeddings are a fundamental component of modern AI systems, including semantic search, Retrieval-Augmented Generation (RAG), recommendation engines, clustering, and AI agents.

----- CHUNKING RESULTS -----
Total semantic chunks: 10

----- EXAMPLE SEMANTIC CHUNK -----
Chunk 5 content (496 characters):
----------------------------------------
| Factor           | Description                          |
| ---------------- | ------------------------------------ |
| Accuracy         | Semantic understanding quality       |
| Dimension Size   | Storage and computation requirements |
| Latency          | Embedding generation speed           |
| Cost             | API or inf

# Recursive Chunking

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_core.documents import Document
import re

In [47]:
from importlib import metadata


def perform_code_chunking(code_document, language="python", chunk_size=100, chunked_overlap=15):

    if language.lower() == "python":
        code_splitter = RecursiveCharacterTextSplitter.from_language(
            language=Language.PYTHON,
            chunk_size=chunk_size,
            chunk_overlap=chunked_overlap
        )
    
    code_chunks = code_splitter.split_text(code_document)
    print(f"code document split into {len(code_chunks)} chunks")

    documents = []

    for chunk in code_chunks:
        # Try to identify code structure
        function_match = re.search(r'def\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*\(', chunk)
        class_match = re.search(r'class\s+([a-zA-Z_][a-zA-Z0-9_]*)', chunk)
        import_match = re.search(r'import\s+([a-zA-Z_][a-zA-Z0-9_\.]*)', chunk)

        chunk_type = "code_segment"
        if function_match:
            chunk_type = "function"
            structure_name = function_match.group(1)
        elif class_match:
            chunk_type = "class"
            structure_name = class_match.group(1)
        elif import_match:
            structure_name = import_match.group(1)
        else: 
            structure_name = f"segment"
        
        doc = Document(
            page_content=chunk,
            metadata = {
                
                "chunk_id": uuid.uuid4(),
                "total_chunks": len(code_chunks),
                "language": language,
                "chunk_type": chunk_type,
                "structure_name": structure_name,
                "lines": chunk.count('\n')+1
            }
        )
        documents.append(doc)
    return documents

In [41]:
python_document = """
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load and prepare data
def load_data(filepath):
    \"\"\"
    Load data from CSV file
    
    Args:
        filepath: Path to the CSV file
        
    Returns:
        Pandas DataFrame containing the data
    \"\"\"
    df = pd.read_csv(filepath)
    print(f"Loaded data with {df.shape[0]} rows and {df.shape[1]} columns")
    return df

def preprocess_data(df, target_column):
    \"\"\"
    Preprocess the data for training
    
    Args:
        df: Input DataFrame
        target_column: Name of the target column
        
    Returns:
        X, y for model training
    \"\"\"
    # Handle missing values
    df = df.fillna(df.mean())
    
    # Split features and target
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    
    return X, y

class ModelTrainer:
    \"\"\"
    Class to handle model training and evaluation
    \"\"\"
    def __init__(self, model_type='rf', random_state=42):
        \"\"\"Initialize the trainer\"\"\"
        self.random_state = random_state
        if model_type == 'rf':
            self.model = RandomForestClassifier(random_state=random_state)
        else:
            raise ValueError(f"Unsupported model type: {model_type}")
    
    def train(self, X, y, test_size=0.2):
        \"\"\"Train the model with train-test split\"\"\"
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state
        )
        
        # Train the model
        self.model.fit(X_train, y_train)
        
        # Evaluate
        train_preds = self.model.predict(X_train)
        test_preds = self.model.predict(X_test)
        
        train_acc = accuracy_score(y_train, train_preds)
        test_acc = accuracy_score(y_test, test_preds)
        
        print(f"Training accuracy: {train_acc:.4f}")
        print(f"Testing accuracy: {test_acc:.4f}")
        
        return {
            'model': self.model,
            'X_test': X_test,
            'y_test': y_test,
            'test_acc': test_acc
        }
    
    def get_feature_importance(self, feature_names):
        \"\"\"Get feature importance from the model\"\"\"
        if not hasattr(self.model, 'feature_importances_'):
            raise ValueError("Model doesn't have feature importances")
        
        importances = self.model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        result = []
        for i in indices:
            result.append({
                'feature': feature_names[i],
                'importance': importances[i]
            })
        
        return result

# Main execution
if __name__ == "__main__":
    # Example usage
    filepath = "data/dataset.csv"
    df = load_data(filepath)
    
    X, y = preprocess_data(df, target_column="target")
    
    trainer = ModelTrainer(model_type='rf')
    results = trainer.train(X, y, test_size=0.25)
    
    # Print feature importance
    importances = trainer.get_feature_importance(X.columns)
    print("\\nFeature Importance:")
    for item in importances[:5]:
        print(f"- {item['feature']}: {item['importance']:.4f}")
"""

In [ ]:
chunked_docs = perform_code_chunking(
    python_document,
    language="python",
    chunk_size=100,
    chunked_overlap=15
)

chunked_docs[0].page_content


code document split into 49 chunks


'import numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split'

# Context-Enriched Chunking

In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
import numpy as np

In [63]:
def perform_context_enriched_chunking(documents, chunk_size=500, chunk_overlap=50, window_size=1, summarize=True):

    llm = ChatOllama(model="llama3", temperature=0, max_token=250)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    base_chunks = splitter.split_text(documents)
    print(f"Document split into {len(base_chunks)} base chunks")

    summary_prompt = PromptTemplate.from_template(
        template="""Provide a brief summary of the following text:\n\n{text}\n\n 
        Summary:
        
        only give summary no additinal details
        """
    )

    chain = summary_prompt | llm

    enriched_documents = []

    for i, chunk in enumerate(base_chunks):
        print(f"Processing chunk {i+1}/{len(base_chunks)}")

        context_summary = chain.invoke({"text": chunk})
        metadata = {
            "context_type" : "summary"
        }
        enriched_text = f"context: {context_summary.content}\n\n Context: {chunk}"

        doc = Document(
            page_content=enriched_text,
            metadata=metadata

        )
        enriched_documents.append(doc)
    
    return enriched_documents



In [64]:
en_doc = perform_context_enriched_chunking(documents=text)

en_doc

Document split into 10 base chunks
Processing chunk 1/10
Processing chunk 2/10
Processing chunk 3/10
Processing chunk 4/10
Processing chunk 5/10
Processing chunk 6/10
Processing chunk 7/10
Processing chunk 8/10
Processing chunk 9/10
Processing chunk 10/10


[Document(metadata={'context_type': 'summary'}, page_content='context: Embeddings are numerical vector representations of data that enable machine learning models to understand semantic meaning and relationships.\n\n Context: # Python for Embeddings in AI Applications\n\n## Introduction\n\nEmbeddings are numerical vector representations of data such as text, images, audio, or documents. They enable machine learning models to understand semantic meaning and relationships between different pieces of information. Embeddings are a fundamental component of modern AI systems, including semantic search, Retrieval-Augmented Generation (RAG), recommendation engines, clustering, and AI agents.'),
 Document(metadata={'context_type': 'summary'}, page_content='context: Python has become the preferred language for working with embeddings due to its rich ecosystem of AI and machine learning libraries. Embeddings convert unstructured data into dense vectors, where similar data points produce close vec